# Production-Ready CLP Label Analyzer

This notebook uses a robust, multi-stage detection system to analyze product labels for CLP compliance.

**Key improvements:**
- Multi-stage detection (rough → refine → validate)
- Handles irregular shapes (not just rectangles)
- Confidence scoring for all detections
- Better error handling and logging
- Production-ready architecture

## Setup & Configuration

In [ ]:
# Install required packages
!pip install -q google-genai pillow pymupdf pydantic

# Import the production analyzer
import sys
sys.path.insert(0, '/Users/clawdy/Desktop')

from label_analyzer_production import (
    LabelAnalyzer,
    analyze_image_file,
    pdf_to_image,
    image_to_base64,
    PartClassification
)

from IPython.display import JSON, Image, Markdown, display
from PIL import Image as PIL_Image
import json
import os

In [ ]:
# Configuration
PROJECT_ID = "hmcp-tst-veraitst-de-prj-arg"
DPI = 300

# File paths
FILE_PATH_IN = "../data/in"
FILE_PATH_OUT = "../data/out"

# Choose which artwork to analyze (uncomment one)
ARTWORK_FILE = "30652435_1"
# ARTWORK_FILE = "30678229_1"
# ARTWORK_FILE = "30660179_2"
# ARTWORK_FILE = "30690350_1"

# Input validation
assert os.path.exists(FILE_PATH_IN), f"Input directory not found: {FILE_PATH_IN}"
os.makedirs(FILE_PATH_OUT, exist_ok=True)

print(f"✓ Configuration loaded")
print(f"  Project: {PROJECT_ID}")
print(f"  Analyzing: {ARTWORK_FILE}")
print(f"  DPI: {DPI}")

## Step 1: Load Image

In [ ]:
# Load PDF and convert to image
import fitz
from io import BytesIO

pdf_path = os.path.join(FILE_PATH_IN, ARTWORK_FILE + ".pdf")
print(f"Loading: {pdf_path}")

# Convert PDF to image at specified DPI
doc = fitz.open(pdf_path)
page = doc.load_page(0)  # First page
zoom = DPI / 72
matrix = fitz.Matrix(zoom, zoom)
pix = page.get_pixmap(matrix=matrix)

# Save intermediate image
image_output_path = os.path.join(FILE_PATH_OUT, f"{ARTWORK_FILE}_page_1.jpg")
pix.save(image_output_path)

# Load as PIL Image
img = PIL_Image.open(image_output_path)
PIL_Image.MAX_IMAGE_PIXELS = None

width, height = img.size
print(f"✓ Image loaded: {width}x{height}px")

# Display
display(img)

## Step 2: Run Analysis Pipeline

In [ ]:
# Authenticate with Google Cloud (only needed once per session)
!gcloud auth application-default login

In [ ]:
# Convert image to base64 for Gemini
image_data = image_to_base64(img)
print(f"✓ Image encoded for Gemini API")

In [ ]:
# Initialize analyzer
analyzer = LabelAnalyzer(project_id=PROJECT_ID, dpi=DPI)
print(f"✓ Analyzer initialized")

In [ ]:
# Run full analysis (takes ~30-60 seconds)
detected_parts = analyzer.analyze(img, image_data)
print(f"\n✓ Analysis complete: {len(detected_parts)} regions detected")

## Step 3: Results Overview

In [ ]:
# Display results as structured JSON
results = analyzer.to_dict()
display(JSON(results))

In [ ]:
# Summary table
print(f"\n{'Classification':<12} {'Label':<30} {'Confidence':<12} {'Area (px²)'}")
print("-" * 70)

clp_count = 0
non_clp_count = 0

for part in detected_parts:
    area = part.rect.area()
    confidence_pct = f"{part.confidence:.0%}"
    classification = part.classification.value
    
    print(f"{classification:<12} {part.label:<30} {confidence_pct:<12} {area:,}")
    
    if part.classification == PartClassification.CLP:
        clp_count += 1
    else:
        non_clp_count += 1

print("-" * 70)
print(f"Total: {clp_count} CLP regions, {non_clp_count} Non-CLP regions")

## Step 4: Visualize Detections

In [ ]:
# Visualize detected regions
visualized = analyzer.visualize(img)
display(visualized)

# Save visualization
viz_path = os.path.join(FILE_PATH_OUT, f"{ARTWORK_FILE}_analyzed.jpg")
visualized.save(viz_path)
print(f"\n✓ Visualization saved: {viz_path}")

## Step 5: Detailed Region Analysis

For each detected region, show cropped image and detailed characteristics

In [ ]:
def resize_image(img: PIL_Image.Image, height: int = 300) -> PIL_Image.Image:
    """Resize image to target height, preserving aspect ratio"""
    if height is None:
        return img
    
    orig_w, orig_h = img.size
    scale = height / orig_h
    new_size = (int(orig_w * scale), height)
    return img.resize(new_size, PIL_Image.LANCZOS)

# Analyze each region
for i, part in enumerate(detected_parts, 1):
    display(Markdown(f"### Region {i}: {part.classification.value} | {part.label}"))
    display(Markdown(f"**Confidence:** {part.confidence:.0%} | **Area:** {part.rect.area():,} px²"))
    
    # Crop region
    cropped = img.crop((
        part.rect.xmin,
        part.rect.ymin,
        part.rect.xmax,
        part.rect.ymax
    ))
    
    # Resize for display
    cropped_resized = resize_image(cropped, height=300)
    display(cropped_resized)
    
    # Save cropped region
    crop_path = os.path.join(
        FILE_PATH_OUT,
        f"{ARTWORK_FILE}_region_{i:02d}_{part.classification.value}.jpg"
    )
    cropped.save(crop_path)
    
    print(f"Dimensions: {part.rect.width()} × {part.rect.height()} px")
    print(f"Position: ({part.rect.xmin}, {part.rect.ymin}) to ({part.rect.xmax}, {part.rect.ymax})")
    print(f"Saved: {crop_path}\n")

## Step 6: Export Results

In [ ]:
# Export full results to JSON
results_json_path = os.path.join(FILE_PATH_OUT, f"{ARTWORK_FILE}_analysis_results.json")

with open(results_json_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✓ Results exported: {results_json_path}")

# Also export as CSV for spreadsheet analysis
import csv

csv_path = os.path.join(FILE_PATH_OUT, f"{ARTWORK_FILE}_analysis_results.csv")

with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'region_num', 'classification', 'label', 'confidence', 
        'xmin', 'ymin', 'xmax', 'ymax', 'width', 'height', 'area'
    ])
    writer.writeheader()
    
    for i, part in enumerate(detected_parts, 1):
        writer.writerow({
            'region_num': i,
            'classification': part.classification.value,
            'label': part.label,
            'confidence': f"{part.confidence:.2f}",
            'xmin': part.rect.xmin,
            'ymin': part.rect.ymin,
            'xmax': part.rect.xmax,
            'ymax': part.rect.ymax,
            'width': part.rect.width(),
            'height': part.rect.height(),
            'area': part.rect.area()
        })

print(f"✓ CSV exported: {csv_path}")

## Summary

✓ **Analysis Complete**

**Files Generated:**
- `{ARTWORK_FILE}_page_1.jpg` - Original high-res image
- `{ARTWORK_FILE}_analyzed.jpg` - Visualization with detected regions
- `{ARTWORK_FILE}_region_*.jpg` - Individual cropped regions
- `{ARTWORK_FILE}_analysis_results.json` - Full structured results
- `{ARTWORK_FILE}_analysis_results.csv` - Spreadsheet-compatible results

**Next Steps:**
1. Review visualized regions for accuracy
2. Check CSV for metrics (areas, positions, confidence)
3. Validate detected regions against actual label structure
4. Fine-tune confidence thresholds if needed
5. Batch process multiple images using `analyze_image_file()` function